# Aula 12 - Redução de Histórico de Chat com Bloco de Notas do Agente

Este notebook demonstra como gerir o contexto em conversas longas usando o Microsoft Agent Framework. À medida que as conversas crescem, o número de tokens aumenta — eventualmente ultrapassando a janela de contexto do modelo. Abordamos isto com um **padrão de sumarização de contexto** e um **bloco de notas do agente** para memória persistente.

## O que vai aprender:
1. **Por que a Gestão de Contexto é Importante**: Compreender os limites de tokens e janelas de contexto
2. **Agentes Conscientes do Contexto**: Construir agentes que gerem o seu próprio contexto de conversa
3. **Padrão de Sumarização de Contexto**: Usar ferramentas para condensar o histórico da conversa
4. **Bloco de Notas do Agente**: Memória persistente que sobrevive à redução do contexto

## Pré-requisitos:
- Configuração do Azure OpenAI com variáveis de ambiente configuradas
- Compreensão dos conceitos básicos de agente das aulas anteriores


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Porquê a Gestão de Contexto é Importante

Cada LLM tem uma **janela de contexto** finita — o número máximo de tokens que pode processar num único pedido. À medida que uma conversação de múltiplas voltas avança:

- **A contagem de tokens cresce linearmente** com cada mensagem do utilizador e resposta do assistente.
- **Os tokens do prompt dominam o custo** porque todo o histórico é reenviado a cada volta.
- Eventualmente, a conversação **ultrapassa a janela de contexto** e o modelo ou trunca ou dá erro.

### Estratégias para Gerir o Contexto

| Estratégia | Como Funciona | Compromisso |
|---|---|---|
| **Truncamento** | Eliminar as mensagens mais antigas | Perde o contexto inicial |
| **Sumarização** | Condensar mensagens mais antigas numa síntese | Perde algum detalhe, mas mantém pontos chave |
| **Scratchpad / Memória Externa** | Guardar factos chave fora da conversação | Requer chamadas a ferramentas, mas resiste a qualquer redução |

Neste notebook combinamos a **sumarização** com uma **ferramenta de scratchpad** para que o agente possa manter a continuidade mesmo quando o histórico da conversa é condensado.


## Criar um Agente Consciente do Contexto


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulando uma Conversa Longa

Vamos passar por uma conversa de múltiplas interações para ver como o contexto se acumula. O agente deve reter detalhes-chave (preferências, orçamento, datas de viagem) ao longo das interações e demonstrar continuidade.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Repare como o agente mantém o contexto de interações anteriores — sabe sobre o Japão, sushi, templos, fotografia, o orçamento de 3000$, a viagem a solo e o período de abril. Numa conversa curta isto funciona bem, mas à medida que a conversa cresce, reenviar todo o histórico torna-se dispendioso.

Vamos continuar a conversa com mais interações para ver a acumulação de contexto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Padrão de Sumário de Contexto

À medida que a conversa cresce, podemos usar uma **ferramenta de resumo** para condensar o contexto acumulado num formato compacto. O agente chama esta ferramenta para registar preferências-chave para que, mesmo que mensagens mais antigas sejam removidas, a informação essencial seja preservada.

Este padrão é o bloco de construção para uma redução de histórico mais sofisticada:
1. O agente identifica factos-chave da conversa
2. Chama a ferramenta de resumo para os guardar
3. Mensagens mais antigas podem ser removidas em segurança porque o resumo captura o que é importante

Abaixo definimos uma ferramenta `summarize_preferences` que o agente pode chamar para registar um resumo compacto do que aprendeu.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Resumo

Nesta lição aprendeu a gerir o contexto em conversas de agentes de longa duração utilizando o Microsoft Agent Framework:

### Conceitos-chave
- **As janelas de contexto são finitas** — cada token no histórico da conversa tem custo e conta para o limite.
- **Ferramentas de sumarização** permitem ao agente condensar o contexto acumulado em resumos compactos, reduzindo o uso de tokens enquanto preservam a informação essencial.
- **Blocos de notas do agente** fornecem memória externa persistente que sobrevive a qualquer redução da conversa.

### O que construiu
- Um **agente consciente do contexto** que mantém a continuidade em conversas de múltiplas interações
- Uma **ferramenta de sumarização** (`summarize_preferences`) que regista detalhes-chave do utilizador num formato compacto
- Uma **conversa de múltiplas interações** demonstrando retenção do contexto e gestão de mudanças

### Aplicações no mundo real
- **Bots de Atendimento ao Cliente**: recordam preferências em longas sessões de suporte
- **Assistentes Pessoais**: acompanham projetos em curso sem necessidade de reexplicar o contexto
- **Tutors Educativos**: mantêm o progresso do aluno ao longo de muitas interações

### Próximos passos
- Implementar uma ferramenta completa de bloco de notas com persistência baseada em ficheiros
- Adicionar truncamento automático do histórico após sumarização
- Combinar com bases de dados vetoriais para pesquisa semântica de memória
- Construir agentes que possam retomar conversas dias depois com contexto completo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional feita por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
